In [2]:
import requests
import csv
import os

os.makedirs("data", exist_ok=True)

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "ALLSKY_SFC_SW_DWN,T2M,WS2M,RH2M,PRECTOT,CLOUD_AMT",
    "community": "RE",
    "latitude": 11.92237,
    "longitude": 79.60673,
    "start": 20230101,
    "end": 20231231,
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()["properties"]["parameter"]

irr = data["ALLSKY_SFC_SW_DWN"]
temp = data["T2M"]
wind = data["WS2M"]
humidity = data["RH2M"]
rain = data["PRECTOTCORR"]
cloud = data["CLOUD_AMT"]

with open("data/powercut.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow([
        "date","irradiance","temperature",
        "humidity","wind_speed","rainfall",
        "cloud","power_cut"
    ])

    for date in irr:

        ir = irr[date]
        t = temp[date]
        w = wind[date]
        h = humidity[date]
        r = rain[date]
        c = cloud[date]

        # 🔥 improved rule
        if (r > 8 and c > 80) or (w > 10) or (h > 90 and c > 85):
            power_cut = 1
        else:
            power_cut = 0

        writer.writerow([date, ir, t, h, w, r, c, power_cut])

print("⚠️ Power cut dataset created!")

⚠️ Power cut dataset created!


In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib

df = pd.read_csv("data/powercut.csv")

# feature engineering
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month

features = [
    "month",
    "irradiance",
    "temperature",
    "humidity",
    "wind_speed",
    "rainfall",
    "cloud"
]

X = df[features]
y = df["power_cut"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# model
model = RandomForestClassifier(n_estimators=200, max_depth=10)
model.fit(X_train, y_train)

# evaluation
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("\n⚠️ POWER CUT MODEL PERFORMANCE")
print("Accuracy:", round(acc*100, 2), "%")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

joblib.dump(model, "models/powercut.pkl")

print("\n✅ Power cut model trained!")


⚠️ POWER CUT MODEL PERFORMANCE
Accuracy: 97.26 %

Confusion Matrix:
 [[64  0]
 [ 2  7]]

✅ Power cut model trained!


In [7]:
import joblib
import pandas as pd

model = joblib.load("models/powercut.pkl")

# -----------------------------
# INPUT
# -----------------------------
input_data = {
    "month": 6,
    "irradiance": 300,
    "temperature": 28,
    "humidity": 92,
    "wind_speed": 12,
    "rainfall": 15,
    "cloud": 90
}

df = pd.DataFrame([input_data])

# -----------------------------
# PREDICT PROBABILITY
# -----------------------------
proba = model.predict_proba(df)[0]

# class 0 = no power cut
# class 1 = power cut
prob_no_cut = proba[0]
prob_cut = proba[1]

# -----------------------------
# DECISION
# -----------------------------
if prob_cut > 0.5:
    status = "⚠️ POWER CUT LIKELY"
else:
    status = "✅ NORMAL GRID"

# -----------------------------
# OUTPUT
# -----------------------------
print(f"\n🔹 Probability of Power Cut: {round(prob_cut, 3)}")
print(f"🔹 Probability of Normal   : {round(prob_no_cut, 3)}")
print(f"\n{status}")


🔹 Probability of Power Cut: 0.83
🔹 Probability of Normal   : 0.17

⚠️ POWER CUT LIKELY


In [8]:
import joblib
import pandas as pd

model = joblib.load("models/powercut.pkl")

tests = [
    ("☀️ Normal Weather", 2, 3, 30),
    ("🌧️ Heavy Rain", 15, 5, 90),
    ("🌪️ Strong Wind", 0, 12, 60),
    ("⛈️ Storm", 20, 15, 95)
]

for name, rain, wind, cloud in tests:

    df = pd.DataFrame([{
        "month": 6,
        "irradiance": 300,
        "temperature": 28,
        "humidity": 90,
        "wind_speed": wind,
        "rainfall": rain,
        "cloud": cloud
    }])

    pred = model.predict(df)[0]
    proba = model.predict_proba(df)[0]

    print(f"\n{name}")
    print("Result:", "⚠️ Power Cut" if pred else "✅ Normal", proba)


☀️ Normal Weather
Result: ✅ Normal [0.655 0.345]

🌧️ Heavy Rain
Result: ⚠️ Power Cut [0.17 0.83]

🌪️ Strong Wind
Result: ✅ Normal [0.645 0.355]

⛈️ Storm
Result: ⚠️ Power Cut [0.18 0.82]
